In [22]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os

In [2]:

# Set the path to the file you'd like to load
# Para este dataset específico, el archivo se llama 'credit_risk_dataset.csv'
file_path = "loan/loan.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ranadeep/credit-risk-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)
print("First 5 records:", df.head())


/tmp/ipykernel_48984/2200756409.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(
/home/tomasgomez/Github/RedesNeuronales_Trabajo2/.venv/lib64/python3.14/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (19: desc, 55: verification_status_joint) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


First 5 records:         id  member_id  loan_amnt  funded_amnt  funded_amnt_inv        term  \
0  1077501    1296599     5000.0       5000.0           4975.0   36 months   
1  1077430    1314167     2500.0       2500.0           2500.0   60 months   
2  1077175    1313524     2400.0       2400.0           2400.0   36 months   
3  1076863    1277178    10000.0      10000.0          10000.0   36 months   
4  1075358    1311748     3000.0       3000.0           3000.0   60 months   

   int_rate  installment grade sub_grade  ... total_bal_il il_util  \
0     10.65       162.87     B        B2  ...          NaN     NaN   
1     15.27        59.83     C        C4  ...          NaN     NaN   
2     15.96        84.33     C        C5  ...          NaN     NaN   
3     13.49       339.31     C        C1  ...          NaN     NaN   
4     12.69        67.79     B        B5  ...          NaN     NaN   

  open_rv_12m  open_rv_24m max_bal_bc all_util total_rev_hi_lim inq_fi  \
0         NaN      

In [3]:
df.last_credit_pull_d.unique()

<StringArray>
['Jan-2016', 'Sep-2013', 'Jan-2015', 'Sep-2015', 'Dec-2014', 'Aug-2012',
 'Mar-2013', 'Dec-2015', 'Aug-2013', 'Nov-2012',
 ...
 'Feb-2008', 'Jan-2008', 'Dec-2007', 'Jul-2008', 'Oct-2007', 'Sep-2007',
 'Jun-2007', 'May-2007', 'Jul-2007', 'Nov-2007']
Length: 104, dtype: str

In [4]:
# 1. Definir el diccionario de mapeo para 0 y 1
mapping = {
    'Fully Paid': 0,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1,
    'Late (31-120 days)': 1,
    'Does not meet the credit policy. Status:Charged Off': 1
}

# 2. Filtrar únicamente los casos válidos (excluyendo los NA)
df = df[df['loan_status'].isin(mapping.keys())].copy()
# 3. Crear la nueva variable objetivo binaria
df['loan_status'] = df['loan_status'].map(mapping)


In [5]:
columnas_id = ["id", "member_id", "policy_code"]
columnas_fechas = ["last_pymnt_d", "next_pymnt_d", "last_credit_pull_d"]
columnas_alta_cardinalidad = [ "url",  "emp_title", "issue_d", "desc", "title", "zip_code", "addr_state", "earliest_cr_line"]
columnas_fuga_datos = [
    'out_prncp', 
    'out_prncp_inv', 
    'total_pymnt', 
    'total_pymnt_inv', 
    'total_rec_prncp', 
    'total_rec_int', 
    'total_rec_late_fee', 
    'recoveries', 
    'collection_recovery_fee', 
    'last_pymnt_amnt'
]

columnas_a_excluir = columnas_id + columnas_fechas + columnas_alta_cardinalidad + columnas_fuga_datos
df = df.drop(columns=columnas_a_excluir)

In [6]:
# 1. Conteo de nulos por columna
conteo_nulos = df.isnull().sum()

# 2. Porcentaje de nulos respecto al total de filas
porcentaje_nulos = (df.isnull().mean() * 100).round(2)

# 3. Unir ambos resultados en un DataFrame ordenado de mayor a menor porcentaje
df_resumen_nulos = pd.DataFrame({
    'Nulos': conteo_nulos,
    'Porcentaje (%)': porcentaje_nulos
}).sort_values(by='Porcentaje (%)', ascending=False)

print(df_resumen_nulos)


                              Nulos  Porcentaje (%)
verification_status_joint    268527          100.00
dti_joint                    268527          100.00
annual_inc_joint             268527          100.00
open_acc_6m                  268386           99.95
open_il_12m                  268386           99.95
open_il_24m                  268386           99.95
total_bal_il                 268386           99.95
open_il_6m                   268386           99.95
il_util                      268404           99.95
open_rv_12m                  268386           99.95
inq_fi                       268386           99.95
open_rv_24m                  268386           99.95
max_bal_bc                   268386           99.95
all_util                     268386           99.95
inq_last_12m                 268386           99.95
mths_since_rcnt_il           268390           99.95
total_cu_tl                  268386           99.95
mths_since_last_record       233873           87.09
mths_since_l

In [7]:
columnas_con_nulos = [
    'annual_inc_joint',
    'verification_status_joint',
    'dti_joint',
    'il_util',
    'mths_since_rcnt_il',
    'inq_fi',
    'open_rv_24m',
    'open_acc_6m',
    'all_util',
    'inq_last_12m',
    'total_cu_tl',
    'open_il_12m',
    'max_bal_bc',
    'open_il_6m',
    'open_il_24m',
    'open_rv_12m',
    'total_bal_il',
    'desc',
    'mths_since_last_record',
    'mths_since_last_major_derog',
    'mths_since_last_delinq',
    'next_pymnt_d',
]

# Eliminar las columnas del DataFrame (axis=1 indica columnas)
df = df.drop(columns=columnas_con_nulos, errors='ignore')

print(f"Columnas restantes en el DataFrame: {df.shape[1]}")


Columnas restantes en el DataFrame: 30


In [8]:
columnas_con_nulos = [
    'total_rev_hi_lim',
    'tot_cur_bal',
    'tot_coll_amt',
    'emp_length',
    'revol_util',
    'collections_12_mths_ex_med'
]

df = df.dropna(subset=columnas_con_nulos).copy()

In [9]:
# Muestra el número de valores únicos por cada columna de texto o categórica
df.select_dtypes(include=['object', 'category']).nunique()

/tmp/ipykernel_48984/2292819751.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=['object', 'category']).nunique()


term                    2
grade                   7
sub_grade              35
emp_length             11
home_ownership          6
verification_status     3
pymnt_plan              2
purpose                13
initial_list_status     2
application_type        2
dtype: int64

In [10]:
print("emp_length: " + str(list((df['emp_length'].unique()))))
print("grade: " + str(list((df['grade'].unique()))))
print("sub_grade: " + str(list((df['sub_grade'].unique()))))
print("verification_status: " + str(list((df['verification_status'].unique()))))

emp_length: ['10+ years', '5 years', '2 years', '4 years', '9 years', '< 1 year', '8 years', '1 year', '3 years', '7 years', '6 years']
grade: ['B', 'A', 'C', 'E', 'D', 'F', 'G']
sub_grade: ['B5', 'B4', 'A3', 'B2', 'C2', 'B3', 'C1', 'B1', 'A1', 'C3', 'C5', 'E1', 'D2', 'E4', 'A5', 'A4', 'E5', 'D5', 'D1', 'F1', 'C4', 'E2', 'D3', 'G3', 'D4', 'G1', 'F2', 'F4', 'F5', 'G5', 'G4', 'F3', 'A2', 'E3', 'G2']
verification_status: ['Source Verified', 'Verified', 'Not Verified']


In [11]:
# 1. Mapeo para 'grade' (de menor a mayor riesgo, A es mejor que G)
grade_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df['grade'] = df['grade'].map(grade_mapping)

# 2. Mapeo automático/ordenado para 'sub_grade' (A1 a G5)
letras = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
subgrade_mapping = {f"{l}{n}": i for i, (l, n) in enumerate(
    (l, n) for l in letras for n in range(1, 6)
)}
df['sub_grade'] = df['sub_grade'].map(subgrade_mapping)

# 3. Mapeo para 'verification_status' (ordenado por nivel de rigor)
verification_mapping = {'Not Verified': 0, 'Source Verified': 1, 'Verified': 2}
df['verification_status'] = df['verification_status'].map(verification_mapping)


# Mapeo ordenado para 'emp_length' (los nulos se pueden dejar como -1 o NaN)
emp_length_mapping = {
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}

# Aplicar el mapeo conservando el nombre de la columna
df['emp_length'] = df['emp_length'].map(emp_length_mapping)

In [12]:
# Selección de columnas no numéricas:
columns_to_dummy = df.select_dtypes(exclude=['number'])

# Vista de columnas no numéricas.
columns_to_dummy.head()

,term,home_ownership,pymnt_plan,purpose,initial_list_status,application_type
42538,36 months,RENT,n,debt_consolidation,w,INDIVIDUAL
42544,36 months,RENT,n,debt_consolidation,f,INDIVIDUAL
42546,36 months,MORTGAGE,n,debt_consolidation,w,INDIVIDUAL
42549,36 months,MORTGAGE,n,credit_card,w,INDIVIDUAL
42550,36 months,MORTGAGE,n,debt_consolidation,f,INDIVIDUAL


In [13]:
# Función auxiliar (generada con Gemini)
# Codificación con one-hot pero utilizando -1/1 en lugar de 0/1.
def dummy_encode(df, column):
    """
    Encodes a categorical column into dummy variables using -1/1 encoding.

    Args:
        df: The input DataFrame.
        column: The name of the column to encode.

    Returns:
        A DataFrame with the dummy variables.
    """

    dummies = pd.get_dummies(df[column], prefix=column, prefix_sep='_', drop_first=True)
    dummies = dummies.map(lambda x: 1 if x == 1 else -1)  # Convert 0s to -1s
    return pd.concat([df.drop(column, axis=1), dummies], axis=1)

In [14]:
# Aplicación de la función auxiliar para hacer el one-hot encoding:
dummy_encoded = columns_to_dummy.copy()

for column in dummy_encoded.columns:
    dummy_encoded = dummy_encode(dummy_encoded, column)

# Visualización del dataframe recodificado
dummy_encoded.head()

,term_ 60 months,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,pymnt_plan_y,purpose_credit_card,purpose_debt_consolidation,purpose_home_improvement,...,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,initial_list_status_w,application_type_JOINT
42538,-1,-1,-1,-1,-1,1,-1,-1,1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,1,-1
42544,-1,-1,-1,-1,-1,1,-1,-1,1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
42546,-1,1,-1,-1,-1,-1,-1,-1,1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,1,-1
42549,-1,1,-1,-1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,1,-1
42550,-1,1,-1,-1,-1,-1,-1,-1,1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


In [17]:
# Selección de las columnas numéricas:
numeric_cols = df.drop(columns_to_dummy.columns.tolist(), axis=1)

# Display the first few rows of the selected numeric columns
numeric_cols.head()


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,emp_length,annual_inc,verification_status,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim
42538,12000.0,12000.0,12000.0,13.53,407.40,1,9,10,40000.0,1,...,7.0,2.0,5572.0,68.8,32.0,0.0,0.0,15386.0,13605.0,8100.0
42544,3000.0,3000.0,3000.0,12.85,100.87,1,8,10,25000.0,2,...,5.0,2.0,2875.0,54.2,26.0,0.0,0.0,154.0,19530.0,5300.0
42546,28000.0,28000.0,28000.0,7.62,872.52,0,2,5,325000.0,1,...,15.0,0.0,29581.0,54.6,31.0,0.0,0.0,0.0,799592.0,54200.0
42549,24000.0,24000.0,24000.0,13.53,814.80,1,9,10,100000.0,2,...,14.0,0.0,21617.0,76.7,39.0,0.0,0.0,539.0,199834.0,28200.0
42550,8000.0,8000.0,8000.0,10.99,261.88,1,6,2,33000.0,0,...,9.0,1.0,7203.0,34.6,16.0,0.0,0.0,0.0,15949.0,20800.0


In [16]:
numeric_cols_norm = -1 + 2 * (numeric_cols - numeric_cols.min()) / (numeric_cols.max() - numeric_cols.min())
numeric_cols_norm.head()


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,emp_length,annual_inc,verification_status,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim
42538,-0.352941,-0.352941,-0.350954,-0.306295,-0.456512,-0.666667,-0.470588,1.0,-0.992184,0.0,...,-0.840000,-0.733333,-0.993620,-0.845792,-0.594595,-1.0,-1.0,-0.996638,-0.996599,-0.992052
42544,-0.882353,-0.882353,-0.879589,-0.363752,-0.895441,-0.666667,-0.529412,1.0,-0.995632,1.0,...,-0.893333,-0.733333,-0.996708,-0.878516,-0.675676,-1.0,-1.0,-0.999966,-0.995118,-0.994834
42546,0.588235,0.588235,0.588840,-0.805661,0.209505,-1.000000,-0.882353,0.0,-0.926672,0.0,...,-0.626667,-1.000000,-0.966130,-0.877620,-0.608108,-1.0,-1.0,-1.000000,-0.800104,-0.946250
42549,0.352941,0.352941,0.353891,-0.306295,0.126854,-0.666667,-0.470588,1.0,-0.978392,1.0,...,-0.653333,-1.000000,-0.975248,-0.828085,-0.500000,-1.0,-1.0,-0.999882,-0.950042,-0.972082
42550,-0.588235,-0.588235,-0.585903,-0.520913,-0.664886,-0.666667,-0.647059,-0.6,-0.993794,-1.0,...,-0.786667,-0.866667,-0.991753,-0.922448,-0.810811,-1.0,-1.0,-1.000000,-0.996013,-0.979434


In [19]:
datos_num_norm = pd.concat([numeric_cols_norm, dummy_encoded], axis=1)
datos_num_norm.head()

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,emp_length,annual_inc,verification_status,...,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,initial_list_status_w,application_type_JOINT
42538,-0.352941,-0.352941,-0.350954,-0.306295,-0.456512,-0.666667,-0.470588,1.0,-0.992184,0.0,...,-1,-1,-1,-1,-1,-1,-1,-1,1,-1
42544,-0.882353,-0.882353,-0.879589,-0.363752,-0.895441,-0.666667,-0.529412,1.0,-0.995632,1.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
42546,0.588235,0.588235,0.588840,-0.805661,0.209505,-1.000000,-0.882353,0.0,-0.926672,0.0,...,-1,-1,-1,-1,-1,-1,-1,-1,1,-1
42549,0.352941,0.352941,0.353891,-0.306295,0.126854,-0.666667,-0.470588,1.0,-0.978392,1.0,...,-1,-1,-1,-1,-1,-1,-1,-1,1,-1
42550,-0.588235,-0.588235,-0.585903,-0.520913,-0.664886,-0.666667,-0.647059,-0.6,-0.993794,-1.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


In [20]:
datos_num_norm.describe()

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,emp_length,annual_inc,verification_status,...,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,initial_list_status_w,application_type_JOINT
count,193050.000000,193050.000000,193050.000000,193050.000000,193050.000000,193050.000000,193050.000000,193050.000000,193050.000000,193050.000000,...,193050.000000,193050.000000,193050.00000,193050.000000,193050.000000,193050.000000,193050.000000,193050.000000,193050.000000,193050.000000
mean,-0.210972,-0.210975,-0.209607,-0.236540,-0.403430,-0.353245,-0.313117,0.207270,-0.984187,0.042813,...,-0.964299,-0.980648,-0.98706,-0.907185,-0.998612,-0.976089,-0.989505,-0.993763,-0.238964,-0.999969
std,0.485155,0.485150,0.484232,0.376431,0.354412,0.445786,0.389367,0.723796,0.013684,0.826747,...,0.264815,0.195782,0.16035,0.420734,0.052674,0.217371,0.144497,0.111511,0.971031,0.007884
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-1.000000,-1.000000,-1.00000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,-0.588235,-0.588235,-0.585903,-0.508238,-0.657254,-0.666667,-0.588235,-0.400000,-0.990575,-1.000000,...,-1.000000,-1.000000,-1.00000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
50%,-0.342647,-0.342647,-0.340675,-0.258978,-0.480325,-0.333333,-0.352941,0.200000,-0.986438,0.000000,...,-1.000000,-1.000000,-1.00000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
75%,0.117647,0.117647,0.117474,0.009717,-0.210278,0.000000,-0.058824,1.000000,-0.980691,1.000000,...,-1.000000,-1.000000,-1.00000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,1.000000,-1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
carpeta_destino = '../datasets'
os.makedirs(carpeta_destino, exist_ok=True)

# 2. Definir la ruta completa del archivo Excel
ruta = os.path.join(carpeta_destino, 'df_procesado.csv')

# 3. Exportar el DataFrame a Excel (index=False evita guardar el índice numérico)
df.to_csv(ruta, index=False)

print(f"Archivo guardado correctamente en: {ruta}")

Archivo guardado correctamente en: ../datasets/df_procesado.csv
